# Tutorial: Unified SSG Foreign Player Recommendation Model Structured-Only v2

이 노트북은 1~6번 레이어를 하나의 input-output 모델로 통합해서
외인투수 3명, 외인타자 3명을 출력하는 최종 설명용 노트북이다.

기사/뉴스/인터뷰/문장형 데이터는 최종 모델 input에서 제외했다.


## 핵심 아이디어

6개의 분석은 따로 노는 모델이 아니라 하나의 추천 모델 안에 들어가는 feature block이다.

- Layer 1: SSG Need Fit
- Layer 2: KBO Archetype / Tool Process
- Layer 3: Market Realism
- Layer 4: KBO Translation
- Layer 5: Structured Failure Resilience / Risk Penalty
- Layer 6: Unified Ranking


In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "outputs").exists():
    ROOT = ROOT.parent
OUT = ROOT / "outputs" / "tables"

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 140)


In [ ]:
blocks = pd.read_csv(OUT / "unified_recommendation_feature_blocks_structured_only_v2.csv")
blocks


## 최종 후보 출력

아래 표가 통합 모델의 최종 output이다.

단, 이 표는 계약 확정 추천이 아니라 **데이터 기반 Top 3 검토 후보**다.


In [ ]:
top3 = pd.read_csv(OUT / "unified_foreign_recommendations_top3_structured_only_v2.csv")
display_cols = [
    "slot_label",
    "recommendation_rank",
    "player_name",
    "team_or_org",
    "primary_position",
    "age",
    "unified_fit_score",
    "data_mining_reason",
    "structured_risk_check",
]
top3[display_cols]


## 후보 풀이 어떻게 줄어들었는가

전체 후보 풀에서 포지션/역할 hard gate를 통과한 선수만 Top 3에 들어간다.


In [ ]:
pool = pd.read_csv(OUT / "unified_foreign_recommendation_pool_structured_only_v2.csv")
pool.groupby(["slot_label", "hard_gate"]).size().reset_index(name="rows")


In [ ]:
pool.sort_values(["slot_label", "unified_fit_score"], ascending=[True, False]).groupby("slot_label").head(8)[
    ["slot_label", "player_name", "primary_position", "unified_fit_score", "hard_gate", "role_or_position_note", "structured_risk_check"]
]


## 발표용 해석

이 모델은 "좋은 선수 순위표"가 아니라 "SSG의 숨은 문제에 맞고, KBO에서 통할 가능성이 있으며, 실제 시장에서 접근 가능하고, 구조화된 실패 리스크가 과도하지 않은 선수"를 찾는다.

기사/뉴스/인터뷰 텍스트는 최종 점수에 사용하지 않는다.

그래서 final score는 다음 네 가지를 동시에 반영한다.

1. SSG에게 필요한 능력인가?
2. KBO로 번역될 가능성이 있는가?
3. 실제로 데려올 수 있는 시장인가?
4. 실패 리스크가 감당 가능한가?
